In [ ]:
import pandas as pd

In [ ]:
nombres_clases = ['World (1)', 'Sports (2)', 'Business (3)', 'Sci/Tech (4)']

## Carga y Preprocesado

In [ ]:
from src.config import PATH_DATA # Ruta para buscar los datos. Por defecto, carpeta data/.

from src.preprocesado import cargar_datos
# from src.preprocesado import preprocessing_avanzado, preprocessing_basico, preprocessing_minimal

In [ ]:
df_train = cargar_datos()

## Vectorizaciones

In [ ]:
from src.config import PATH_MODELS # Ruta donde se guardan/leen los modelos.
PATH_MODELS

### Dispersas

In [ ]:
from src.vectorizacion import BoW, TF_IDF

In [ ]:
df_bow, _ = BoW(df_train.desc_avanzado)
df_tfidf, _ = TF_IDF(df_train.desc_basico)

In [ ]:
print("Palabras más comunes en BoW")
print(df_bow.sum(axis=0).sort_values(ascending=False)[:5])
print()
print("Palabras más representativas en TF-IDF")
print(df_tfidf.sum(axis=0).sort_values(ascending=False)[:5])

### Densas

In [ ]:
from src.vectorizacion import get_word2vec_model
from src.vectorizacion import Word2Vec_Embeddings, BERT_Embeddings

In [ ]:
w2v = get_word2vec_model()

In [ ]:
muestra = df_train.head(5)

print("Word2Vec")

df_w2v_muestra = Word2Vec_Embeddings(muestra['desc_basico'], w2v)

print(f"Forma de la matriz: {df_w2v_muestra.shape}") 
display(df_w2v_muestra.head(2)) 


print("BERT")
df_bert_muestra = BERT_Embeddings(muestra['desc_minimal'])

print(f"Forma de la matriz: {df_bert_muestra.shape}")
display(df_bert_muestra.head(2))

## Clasificadores

In [ ]:
df_test = cargar_datos(test=True)
df_test

### Clasificador con BoW + Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
X_train_bow, vectorizador_bow = BoW(df_train['desc_avanzado'], max_features=3000)
X_test_bow,_ = BoW(df_test['desc_avanzado'], max_features=3000,vectorizador=vectorizador_bow)

y_train = df_train['label']
y_test = df_test['label']

In [ ]:
print(f"- Dimensiones Train: {X_train_bow.shape}")
print(f"- Dimensiones Test:  {X_test_bow.shape}")

# Modelo de Naive Bayes
modelo_nb = MultinomialNB()
modelo_nb.fit(X_train_bow, y_train)

# Predicción
y_pred_bow = modelo_nb.predict(X_test_bow)

print("-"*30)
print("- Resultados de Naive Bayes")
print(f"- Accuracy Global: {accuracy_score(y_test, y_pred_bow):.4f}\n")
print("- Desglose por temática:")
print()
print(classification_report(y_test, y_pred_bow, target_names=nombres_clases))

### Clasificación con TF-IDF + Regresor Logístico

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
X_train_tfidf, vectorizador_tfidf = TF_IDF(df_train['desc_basico'],max_features=3000)
X_test_tfidf,_ = TF_IDF(df_test['desc_basico'],max_features=3000, vectorizador=vectorizador_tfidf)

y_train = df_train['label']
y_test = df_test['label']

In [ ]:
print(f"- Dimensiones Train: {X_train_tfidf.shape}")
print(f"- Dimensiones Test:  {X_test_tfidf.shape}")

# Modelo de Regresor Logístico
modelo_lr = LogisticRegression(max_iter=1000, random_state=123)
modelo_lr.fit(X_train_tfidf, y_train)

# Predicción
y_pred_tfidf = modelo_lr.predict(X_test_tfidf)

print("-"*30)
print("- Resultados de Regresor Logístico")
print(f"- Accuracy Global: {accuracy_score(y_test, y_pred_tfidf):.4f}\n")
print("- Desglose por temática:")
print(classification_report(y_test, y_pred_tfidf, target_names=nombres_clases))

### Clasificación con Word2Vec + RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
w2v = get_word2vec_model()

In [ ]:
X_train_w2v = Word2Vec_Embeddings(df_train['desc_basico'], w2v)
X_test_w2v  = Word2Vec_Embeddings(df_test['desc_basico'], w2v)

y_train = df_train['label']
y_test = df_test['label']

In [ ]:
print(f"- Dimensiones Train: {X_train_tfidf.shape}")
print(f"- Dimensiones Test:  {X_test_tfidf.shape}")

# Modelo de RandomForest
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=123, n_jobs=-1)
modelo_rf.fit(X_train_w2v, y_train)

# Predicción
y_pred_w2v = modelo_rf.predict(X_test_w2v)

print("-"*30)
print("- Resultados de RandomForest")
print(f"- Accuracy Global: {accuracy_score(y_test, y_pred_w2v):.4f}\n")
print()
print("- Desglose por temática:")
print(classification_report(y_test, y_pred_w2v, target_names=nombres_clases))

### Clasificador con BERT + MLP

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
ruta_train_bert = PATH_DATA / "train_bert.parquet"
ruta_test_bert  = PATH_DATA / "test_bert.parquet"

y_train = df_train['label']
y_test = df_test['label']

if ruta_train_bert.exists() and ruta_test_bert.exists():
    print("Cargando Embeddings desde almacenamiento...")
    X_train_bert = pd.read_parquet(ruta_train_bert)
    X_test_bert  = pd.read_parquet(ruta_test_bert)
    
else:
    print("Creando Embeddings desde 0...")
    X_train_bert = BERT_Embeddings(df_train['desc_minimal'])
    X_test_bert  = BERT_Embeddings(df_test['desc_minimal'])
    
    X_train_bert.to_parquet(ruta_train_bert, engine="pyarrow", index=False)
    X_test_bert.to_parquet(ruta_test_bert, engine="pyarrow", index=False)


In [ ]:
print(f"- Dimensiones Train: {X_train_bert.shape}")
print(f"- Dimensiones Test:  {X_test_bert.shape}")

modelo_mlp = MLPClassifier(
    hidden_layer_sizes=(128,), 
    activation='relu', 
    solver='adam', 
    early_stopping=True, 
    max_iter=300, 
    random_state=123
)

modelo_mlp.fit(X_train_bert, y_train)

# Predicción
y_pred_bert = modelo_mlp.predict(X_test_bert)

print("-"*30)
print(f"- Accuracy Global: {accuracy_score(y_test, y_pred_bert):.4f}\n")
print()
print("- Desglose por temática:")
print(classification_report(y_test, y_pred_bert, target_names=nombres_clases))

## Fine-Tunning